# Análisis de Ventas de Videojuegos - Tienda Ice

## Descripción del Proyecto

Este proyecto analiza datos históricos de ventas de videojuegos para identificar patrones que determinen el éxito de un juego. El objetivo es detectar proyectos prometedores y planificar campañas publicitarias para 2017.

**Fecha de análisis:** Diciembre 2016  
**Objetivo:** Planificar campaña para 2017

---

## Paso 1: Importación de Librerías y Carga de Datos

In [ ]:
# Importar las librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Configuración de pandas para mostrar más filas y columnas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [ ]:
# Cargar los datos
# Nota: Asegúrate de tener el archivo games.csv en la ruta correcta
df = pd.read_csv('/datasets/games.csv')

# Si el archivo no está disponible, descargarlo:
# df = pd.read_csv('https://practicum-content.s3.us-west-1.amazonaws.com/datasets/games.csv')

print("Dataset cargado exitosamente")
print(f"Forma del dataset: {df.shape}")

In [ ]:
# Exploración inicial de los datos
print("Primeras 10 filas del dataset:")
df.head(10)

In [ ]:
# Información general del dataset
print("Información general del dataset:")
df.info()

In [ ]:
# Estadísticas descriptivas
print("Estadísticas descriptivas:")
df.describe()

In [ ]:
# Verificar valores únicos en columnas categóricas
print("Número de plataformas únicas:", df['Platform'].nunique())
print("Número de géneros únicos:", df['Genre'].nunique())
print("\nRatings ESRB disponibles:")
print(df['Rating'].value_counts(dropna=False))

## Paso 2: Preparación de Datos

### 2.1 Normalización de Nombres de Columnas

In [ ]:
# Convertir nombres de columnas a minúsculas
df.columns = df.columns.str.lower()
print("Nombres de columnas después de la normalización:")
print(df.columns.tolist())

### 2.2 Conversión de Tipos de Datos

In [ ]:
# Verificar tipos de datos actuales
print("Tipos de datos antes de la conversión:")
print(df.dtypes)
print("\n" + "="*50 + "\n")

In [ ]:
# Convertir year_of_release a tipo numérico
# Primero revisar valores únicos para detectar problemas
print("Valores únicos en year_of_release:")
print(df['year_of_release'].value_counts(dropna=False).head(20))

In [ ]:
# Convertir year_of_release a entero (int)
df['year_of_release'] = pd.to_numeric(df['year_of_release'], errors='coerce')
df['year_of_release'] = df['year_of_release'].astype('Int64')  # Int64 permite valores nulos

print("✓ year_of_release convertido a Int64")
print(f"  Razón: Esta columna representa años y debe ser un número entero.")
print(f"  Los valores no numéricos se convirtieron a NaN.\n")

In [ ]:
# Convertir user_score a numérico
# Primero revisar los valores únicos
print("Valores únicos en user_score (primeros 20):")
print(df['user_score'].value_counts(dropna=False).head(20))

In [ ]:
# Manejar el valor 'tbd' (to be determined) en user_score
# Reemplazar 'tbd' con NaN
df['user_score'] = df['user_score'].replace('tbd', np.nan)

# Convertir a float
df['user_score'] = pd.to_numeric(df['user_score'], errors='coerce')

print("✓ user_score convertido a float64")
print(f"  Razón: Esta columna contiene calificaciones numéricas (máximo 10).")
print(f"  Los valores 'tbd' fueron reemplazados por NaN porque representan")
print(f"  puntuaciones que aún no han sido determinadas.\n")

In [ ]:
# Verificar tipos de datos después de la conversión
print("Tipos de datos después de la conversión:")
print(df.dtypes)

### 2.3 Análisis de Valores Ausentes

In [ ]:
# Contar valores ausentes por columna
missing_values = df.isnull().sum()
missing_percentage = (missing_values / len(df)) * 100

missing_df = pd.DataFrame({
    'Columna': missing_values.index,
    'Valores Ausentes': missing_values.values,
    'Porcentaje': missing_percentage.values
})

missing_df = missing_df[missing_df['Valores Ausentes'] > 0].sort_values('Valores Ausentes', ascending=False)

print("Valores ausentes por columna:")
print(missing_df.to_string(index=False))

In [ ]:
# Visualizar valores ausentes
plt.figure(figsize=(12, 6))
missing_data = df.isnull().sum().sort_values(ascending=False)
missing_data = missing_data[missing_data > 0]

plt.bar(range(len(missing_data)), missing_data.values)
plt.xticks(range(len(missing_data)), missing_data.index, rotation=45, ha='right')
plt.title('Valores Ausentes por Columna', fontsize=14, fontweight='bold')
plt.ylabel('Cantidad de Valores Ausentes')
plt.xlabel('Columna')
plt.tight_layout()
plt.show()

### Estrategia para Manejo de Valores Ausentes

**Explicación de valores ausentes:**

1. **name**: Muy pocos valores ausentes. Estos registros pueden representar juegos sin nombre oficial o datos incompletos de entrada. Se mantienen como NaN.

2. **year_of_release**: Aproximadamente 269 valores ausentes. Posibles razones:
   - Juegos muy antiguos sin registro de fecha
   - Errores en la captura de datos
   - Juegos cancelados o sin lanzamiento oficial
   
3. **critic_score** y **user_score**: Gran cantidad de valores ausentes (~50-60%).
   - Juegos lanzados recientemente sin suficientes reseñas
   - Juegos de nicho o indie con poca cobertura
   - Juegos antiguos lanzados antes de las plataformas de reseñas
   
4. **rating**: ~40% de valores ausentes.
   - Juegos lanzados en regiones sin clasificación ESRB
   - Juegos muy antiguos sin sistema de clasificación
   - Juegos digitales o indie sin clasificación oficial

**Decisión de manejo:**
- Se **mantienen los valores NaN** en lugar de eliminar filas o rellenar con valores arbitrarios
- Los análisis se harán con `dropna=True` cuando sea necesario
- Esto preserva la integridad de los datos y evita sesgos artificiales

### 2.4 Cálculo de Ventas Totales

In [ ]:
# Calcular ventas totales por juego
df['total_sales'] = df['na_sales'] + df['eu_sales'] + df['jp_sales'] + df['other_sales']

print("Columna 'total_sales' creada exitosamente")
print(f"\nEstadísticas de ventas totales:")
print(df['total_sales'].describe())

In [ ]:
# Verificar los juegos con mayores ventas totales
print("Top 10 juegos con mayores ventas totales:")
top_games = df.nlargest(10, 'total_sales')[['name', 'platform', 'year_of_release', 'genre', 'total_sales']]
print(top_games.to_string(index=False))

## Paso 3: Análisis de Datos

### 3.1 Juegos Lanzados por Año

In [ ]:
# Análisis de lanzamientos por año
games_per_year = df.groupby('year_of_release').size().reset_index(name='count')
games_per_year = games_per_year.dropna()
games_per_year['year_of_release'] = games_per_year['year_of_release'].astype(int)
games_per_year = games_per_year.sort_values('year_of_release')

print("Juegos lanzados por año:")
print(games_per_year.to_string(index=False))

In [ ]:
# Visualización de lanzamientos por año
plt.figure(figsize=(14, 6))
plt.plot(games_per_year['year_of_release'], games_per_year['count'], marker='o', linewidth=2, markersize=6)
plt.fill_between(games_per_year['year_of_release'], games_per_year['count'], alpha=0.3)
plt.title('Número de Juegos Lanzados por Año', fontsize=14, fontweight='bold')
plt.xlabel('Año')
plt.ylabel('Número de Juegos')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Observaciones sobre los datos por período:**

- Los datos anteriores a 2000 son muy limitados y poco representativos
- Hay un crecimiento significativo en el número de lanzamientos desde 2000
- El pico de lanzamientos ocurre alrededor de 2008-2010
- Los datos de 2016 están incompletos (solo hasta diciembre)
- Para construir un modelo predictivo para 2017, **consideraremos datos desde 2013-2016** (últimos 4 años), ya que representan las tendencias más recientes del mercado

### 3.2 Análisis de Ventas por Plataforma

In [ ]:
# Ventas totales por plataforma
platform_sales = df.groupby('platform')['total_sales'].sum().sort_values(ascending=False)

print("Ventas totales por plataforma (Top 15):")
print(platform_sales.head(15))

In [ ]:
# Visualización de ventas totales por plataforma
plt.figure(figsize=(14, 6))
platform_sales.head(15).plot(kind='barh', color='steelblue')
plt.title('Top 15 Plataformas por Ventas Totales', fontsize=14, fontweight='bold')
plt.xlabel('Ventas Totales (millones USD)')
plt.ylabel('Plataforma')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Evolución de ventas por plataforma a lo largo del tiempo
# Seleccionar las top 8 plataformas
top_platforms = platform_sales.head(8).index.tolist()

platform_year_sales = df[df['platform'].isin(top_platforms)].groupby(
    ['year_of_release', 'platform']
)['total_sales'].sum().reset_index()

platform_year_sales = platform_year_sales.dropna()

# Visualización
plt.figure(figsize=(16, 8))
for platform in top_platforms:
    data = platform_year_sales[platform_year_sales['platform'] == platform]
    plt.plot(data['year_of_release'], data['total_sales'], marker='o', label=platform, linewidth=2)

plt.title('Evolución de Ventas por Plataforma (Top 8)', fontsize=14, fontweight='bold')
plt.xlabel('Año')
plt.ylabel('Ventas Totales (millones USD)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Observaciones sobre ciclos de vida de plataformas:**

- **Plataformas en declive**: PS2, Wii, X360, PS3 muestran ventas decrecientes
- **Plataformas emergentes**: PS4, XOne están en crecimiento
- **Ciclo de vida típico**: Una plataforma tarda aproximadamente 2-3 años en alcanzar su pico de ventas desde el lanzamiento, y luego declina durante 3-5 años hasta ser reemplazada
- **Datos relevantes para 2017**: Se deben considerar principalmente los datos de 2013-2016

### 3.3 Filtrado de Datos Relevantes

In [ ]:
# Filtrar datos relevantes (2013-2016) para el análisis predictivo
df_relevant = df[df['year_of_release'] >= 2013].copy()

print(f"Dataset original: {len(df)} registros")
print(f"Dataset relevante (2013-2016): {len(df_relevant)} registros")
print(f"Porcentaje de datos mantenidos: {len(df_relevant)/len(df)*100:.2f}%")

### 3.4 Plataformas Líderes en el Período Relevante

In [ ]:
# Análisis de plataformas en el período relevante
platform_sales_recent = df_relevant.groupby('platform')['total_sales'].sum().sort_values(ascending=False)

print("Top 10 plataformas por ventas (2013-2016):")
print(platform_sales_recent.head(10))

In [ ]:
# Análisis de crecimiento por plataforma
# Comparar ventas 2015 vs 2016
sales_2015 = df[df['year_of_release'] == 2015].groupby('platform')['total_sales'].sum()
sales_2016 = df[df['year_of_release'] == 2016].groupby('platform')['total_sales'].sum()

growth_df = pd.DataFrame({
    'sales_2015': sales_2015,
    'sales_2016': sales_2016
}).fillna(0)

growth_df['growth'] = ((growth_df['sales_2016'] - growth_df['sales_2015']) / 
                        growth_df['sales_2015'].replace(0, 1)) * 100

growth_df = growth_df[growth_df['sales_2015'] > 10]  # Filtrar plataformas con ventas significativas
growth_df = growth_df.sort_values('growth', ascending=False)

print("Crecimiento de plataformas (2015 vs 2016):")
print(growth_df.head(10))

**Plataformas potencialmente rentables para 2017:**

1. **PS4**: Líder en ventas con tendencia de crecimiento
2. **XOne**: Segunda posición con base sólida
3. **3DS**: Fuerte en mercado portátil
4. **PC**: Plataforma estable con mercado digital en expansión
5. **WiiU**: A pesar de ventas moderadas, tiene nicho específico

### 3.5 Diagrama de Caja - Ventas Globales por Plataforma

In [ ]:
# Boxplot de ventas por plataforma (top platforms)
top_platforms_recent = platform_sales_recent.head(10).index.tolist()

plt.figure(figsize=(14, 8))
data_to_plot = [df_relevant[df_relevant['platform'] == p]['total_sales'].dropna() 
                for p in top_platforms_recent]

box = plt.boxplot(data_to_plot, labels=top_platforms_recent, patch_artist=True)

# Colorear las cajas
colors = plt.cm.Set3(range(len(top_platforms_recent)))
for patch, color in zip(box['boxes'], colors):
    patch.set_facecolor(color)

plt.title('Distribución de Ventas Globales por Plataforma (2013-2016)', fontsize=14, fontweight='bold')
plt.xlabel('Plataforma')
plt.ylabel('Ventas Totales (millones USD)')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Calcular estadísticas por plataforma
platform_stats = df_relevant.groupby('platform')['total_sales'].agg([
    ('media', 'mean'),
    ('mediana', 'median'),
    ('std', 'std'),
    ('total', 'sum'),
    ('count', 'count')
]).round(2)

platform_stats = platform_stats.sort_values('total', ascending=False).head(10)

print("Estadísticas de ventas por plataforma:")
print(platform_stats)

**Hallazgos sobre diferencias en ventas:**

- **Diferencias significativas**: Las plataformas muestran diferencias notables tanto en ventas totales como en distribución
- **Ventas promedio**: PS4 y XOne tienen ventas promedio más altas por juego
- **Variabilidad**: Todas las plataformas muestran alta variabilidad (desviación estándar), indicando que algunos juegos tienen ventas excepcionales mientras otros son modestas
- **Outliers**: Los diagramas de caja muestran muchos valores atípicos, representando juegos "blockbuster" que superan significativamente las ventas típicas

### 3.6 Correlación entre Reseñas y Ventas

In [ ]:
# Elegir PS4 como plataforma popular para el análisis
ps4_data = df_relevant[df_relevant['platform'] == 'PS4'].copy()

print(f"Juegos de PS4 en el dataset relevante: {len(ps4_data)}")
print(f"Juegos con critic_score: {ps4_data['critic_score'].notna().sum()}")
print(f"Juegos con user_score: {ps4_data['user_score'].notna().sum()}")

In [ ]:
# Gráfico de dispersión: Critic Score vs Ventas
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Critic Score vs Sales
ps4_critic = ps4_data.dropna(subset=['critic_score', 'total_sales'])
axes[0].scatter(ps4_critic['critic_score'], ps4_critic['total_sales'], alpha=0.6, s=50)
axes[0].set_title('PS4: Critic Score vs Ventas Totales', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Critic Score (0-100)')
axes[0].set_ylabel('Ventas Totales (millones USD)')
axes[0].grid(True, alpha=0.3)

# Calcular correlación
corr_critic = ps4_critic['critic_score'].corr(ps4_critic['total_sales'])
axes[0].text(0.05, 0.95, f'Correlación: {corr_critic:.3f}', 
             transform=axes[0].transAxes, fontsize=11, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# User Score vs Sales
ps4_user = ps4_data.dropna(subset=['user_score', 'total_sales'])
axes[1].scatter(ps4_user['user_score'], ps4_user['total_sales'], alpha=0.6, s=50, color='coral')
axes[1].set_title('PS4: User Score vs Ventas Totales', fontsize=12, fontweight='bold')
axes[1].set_xlabel('User Score (0-10)')
axes[1].set_ylabel('Ventas Totales (millones USD)')
axes[1].grid(True, alpha=0.3)

# Calcular correlación
corr_user = ps4_user['user_score'].corr(ps4_user['total_sales'])
axes[1].text(0.05, 0.95, f'Correlación: {corr_user:.3f}', 
             transform=axes[1].transAxes, fontsize=11, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

In [ ]:
# Análisis estadístico de correlaciones
print("ANÁLISIS DE CORRELACIÓN - PS4")
print("="*50)
print(f"\nCorrelación Critic Score vs Ventas: {corr_critic:.4f}")
print(f"Correlación User Score vs Ventas: {corr_user:.4f}")
print("\nInterpretación:")
print("- Correlación positiva moderada indica que mejores reseñas")
print("  tienden a asociarse con mayores ventas")
print("- Sin embargo, la correlación no es perfecta, sugiriendo que")
print("  otros factores (marketing, franquicia, etc.) también influyen")

**Conclusiones sobre la relación reseñas-ventas:**

- Existe una correlación positiva moderada entre las reseñas de críticos/usuarios y las ventas
- Las reseñas de críticos suelen tener una correlación ligeramente mayor que las de usuarios
- La correlación no es perfecta, lo que indica que las ventas dependen de múltiples factores
- Factores adicionales importantes: reconocimiento de marca, presupuesto de marketing, franquicia establecida

### 3.7 Comparación de Ventas entre Plataformas

In [ ]:
# Analizar juegos multiplataforma
# Encontrar juegos que están en múltiples plataformas
game_counts = df_relevant.groupby('name')['platform'].nunique()
multiplatform_games = game_counts[game_counts > 1].index.tolist()

print(f"Juegos multiplataforma (2013-2016): {len(multiplatform_games)}")
print(f"\nEjemplos de juegos multiplataforma:")
print(multiplatform_games[:10])

In [ ]:
# Comparar ventas de un juego específico en diferentes plataformas
# Seleccionar un juego popular multiplataforma
sample_game = multiplatform_games[0]  # Tomar el primero como ejemplo

game_platforms = df_relevant[df_relevant['name'] == sample_game][['platform', 'total_sales']].sort_values('total_sales', ascending=False)

print(f"Ventas de '{sample_game}' por plataforma:")
print(game_platforms.to_string(index=False))

In [ ]:
# Análisis agregado de ventas por plataforma para juegos multiplataforma
multiplatform_df = df_relevant[df_relevant['name'].isin(multiplatform_games)]

platform_multiplatform_sales = multiplatform_df.groupby('platform')['total_sales'].agg([
    ('promedio', 'mean'),
    ('mediana', 'median'),
    ('total', 'sum'),
    ('cantidad', 'count')
]).round(2)

platform_multiplatform_sales = platform_multiplatform_sales[platform_multiplatform_sales['cantidad'] >= 10]
platform_multiplatform_sales = platform_multiplatform_sales.sort_values('promedio', ascending=False)

print("Ventas promedio de juegos multiplataforma por plataforma:")
print(platform_multiplatform_sales.head(10))

**Hallazgos sobre ventas multiplataforma:**

- Los mismos juegos tienden a vender más en plataformas con mayor base instalada (PS4, XOne)
- Las diferencias en ventas reflejan el tamaño del mercado de cada plataforma
- La calidad del port y el marketing específico por plataforma también influyen

### 3.8 Distribución por Género

In [ ]:
# Análisis de géneros
genre_sales = df_relevant.groupby('genre')['total_sales'].sum().sort_values(ascending=False)
genre_count = df_relevant['genre'].value_counts()

genre_stats = pd.DataFrame({
    'total_ventas': genre_sales,
    'promedio_ventas': df_relevant.groupby('genre')['total_sales'].mean(),
    'num_juegos': genre_count
}).round(2)

genre_stats = genre_stats.sort_values('total_ventas', ascending=False)

print("Estadísticas por género:")
print(genre_stats)

In [ ]:
# Visualización de distribución de géneros
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Ventas totales por género
genre_sales.plot(kind='barh', ax=axes[0], color='teal')
axes[0].set_title('Ventas Totales por Género (2013-2016)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Ventas Totales (millones USD)')
axes[0].set_ylabel('Género')
axes[0].invert_yaxis()

# Ventas promedio por género
genre_avg = df_relevant.groupby('genre')['total_sales'].mean().sort_values(ascending=False)
genre_avg.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Ventas Promedio por Género (2013-2016)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Ventas Promedio (millones USD)')
axes[1].set_ylabel('Género')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

**Conclusiones sobre géneros:**

**Géneros más rentables:**
- **Action**: Líder absoluto en ventas totales
- **Shooter**: Alto volumen y ventas promedio elevadas
- **Sports**: Rendimiento consistente
- **Role-Playing**: Ventas totales altas con base de fans leales

**Géneros con ventas bajas:**
- **Strategy**: Nicho específico, menor audiencia masiva
- **Puzzle**: Mercado limitado, muchos títulos casuales
- **Adventure**: Depende altamente de la franquicia

**Generalización:**
- Géneros orientados a acción y competencia tienden a tener ventas más altas
- Géneros narrativos o de nicho tienen audiencias más pequeñas pero leales

## Paso 4: Perfil de Usuario por Región

### 4.1 Top 5 Plataformas por Región

In [ ]:
# Top 5 plataformas por región
regions = {'na_sales': 'Norteamérica', 'eu_sales': 'Europa', 'jp_sales': 'Japón'}

for region_col, region_name in regions.items():
    print(f"\n{'='*60}")
    print(f"TOP 5 PLATAFORMAS - {region_name.upper()}")
    print(f"{'='*60}")
    
    top_platforms_region = df_relevant.groupby('platform')[region_col].sum().sort_values(ascending=False).head(5)
    
    for i, (platform, sales) in enumerate(top_platforms_region.items(), 1):
        percentage = (sales / top_platforms_region.sum()) * 100
        print(f"{i}. {platform}: {sales:.2f} millones USD ({percentage:.1f}% del top 5)")

In [ ]:
# Visualización comparativa de plataformas por región
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (region_col, region_name) in enumerate(regions.items()):
    top_platforms_region = df_relevant.groupby('platform')[region_col].sum().sort_values(ascending=False).head(5)
    
    axes[idx].barh(range(len(top_platforms_region)), top_platforms_region.values)
    axes[idx].set_yticks(range(len(top_platforms_region)))
    axes[idx].set_yticklabels(top_platforms_region.index)
    axes[idx].set_title(f'Top 5 Plataformas - {region_name}', fontweight='bold')
    axes[idx].set_xlabel('Ventas (millones USD)')
    axes[idx].invert_yaxis()
    axes[idx].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

**Variaciones en cuotas de mercado:**

- **Norteamérica y Europa**: Preferencia similar por PS4 y XOne
- **Japón**: Mercado dominado por plataformas portátiles (3DS, PSV) y PS4
- **XOne**: Muy débil en Japón comparado con occidente
- **Nintendo**: Mayor penetración en Japón (3DS, WiiU)

### 4.2 Top 5 Géneros por Región

In [ ]:
# Top 5 géneros por región
for region_col, region_name in regions.items():
    print(f"\n{'='*60}")
    print(f"TOP 5 GÉNEROS - {region_name.upper()}")
    print(f"{'='*60}")
    
    top_genres_region = df_relevant.groupby('genre')[region_col].sum().sort_values(ascending=False).head(5)
    
    for i, (genre, sales) in enumerate(top_genres_region.items(), 1):
        percentage = (sales / top_genres_region.sum()) * 100
        print(f"{i}. {genre}: {sales:.2f} millones USD ({percentage:.1f}% del top 5)")

In [ ]:
# Visualización comparativa de géneros por región
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (region_col, region_name) in enumerate(regions.items()):
    top_genres_region = df_relevant.groupby('genre')[region_col].sum().sort_values(ascending=False).head(5)
    
    axes[idx].barh(range(len(top_genres_region)), top_genres_region.values, color='steelblue')
    axes[idx].set_yticks(range(len(top_genres_region)))
    axes[idx].set_yticklabels(top_genres_region.index)
    axes[idx].set_title(f'Top 5 Géneros - {region_name}', fontweight='bold')
    axes[idx].set_xlabel('Ventas (millones USD)')
    axes[idx].invert_yaxis()
    axes[idx].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

**Diferencias en preferencias de género:**

- **Norteamérica**: Preferencia por Action, Shooter y Sports
- **Europa**: Similar a NA, con ligeramente más Sports
- **Japón**: Marcada preferencia por Role-Playing, Action menos dominante
- **Shooter**: Muy popular en occidente, menos en Japón
- **RPG**: Significativamente más popular en Japón

### 4.3 Impacto de Clasificación ESRB por Región

In [ ]:
# Análisis de ratings ESRB por región
for region_col, region_name in regions.items():
    print(f"\n{'='*60}")
    print(f"VENTAS POR RATING ESRB - {region_name.upper()}")
    print(f"{'='*60}")
    
    rating_sales = df_relevant.groupby('rating')[region_col].sum().sort_values(ascending=False)
    
    for rating, sales in rating_sales.items():
        print(f"{rating}: {sales:.2f} millones USD")

In [ ]:
# Visualización de ratings por región
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (region_col, region_name) in enumerate(regions.items()):
    rating_sales = df_relevant.groupby('rating')[region_col].sum().sort_values(ascending=False)
    
    axes[idx].bar(range(len(rating_sales)), rating_sales.values, color='coral')
    axes[idx].set_xticks(range(len(rating_sales)))
    axes[idx].set_xticklabels(rating_sales.index, rotation=45, ha='right')
    axes[idx].set_title(f'Ventas por Rating ESRB - {region_name}', fontweight='bold')
    axes[idx].set_ylabel('Ventas (millones USD)')
    axes[idx].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

**Impacto de clasificación ESRB:**

- **Todas las regiones**: Juegos clasificados M (Mature) y E (Everyone) dominan las ventas
- **Norteamérica**: Mayor proporción de juegos M
- **Europa**: Distribución similar a NA
- **Japón**: Muchos juegos sin clasificación ESRB (sistema diferente)
- El rating ESRB **sí afecta las ventas**, con M y E siendo los más comerciales

## Paso 5: Pruebas de Hipótesis

### 5.1 Hipótesis 1: Calificaciones de Xbox One vs PC

**Formulación de hipótesis:**

- **H₀ (Hipótesis nula)**: Las calificaciones promedio de usuarios para Xbox One y PC son iguales (μ_XOne = μ_PC)
- **H₁ (Hipótesis alternativa)**: Las calificaciones promedio de usuarios para Xbox One y PC son diferentes (μ_XOne ≠ μ_PC)

**Nivel de significancia**: α = 0.05

**Criterio**: Prueba t de Student para dos muestras independientes
- Apropiada para comparar medias de dos grupos independientes
- Asumimos distribuciones aproximadamente normales (por teorema del límite central con n suficientemente grande)

In [ ]:
# Preparar datos para la prueba
xone_scores = df_relevant[df_relevant['platform'] == 'XOne']['user_score'].dropna()
pc_scores = df_relevant[df_relevant['platform'] == 'PC']['user_score'].dropna()

print("ESTADÍSTICAS DESCRIPTIVAS")
print("="*60)
print(f"\nXbox One:")
print(f"  Tamaño de muestra: {len(xone_scores)}")
print(f"  Media: {xone_scores.mean():.4f}")
print(f"  Desviación estándar: {xone_scores.std():.4f}")
print(f"  Mediana: {xone_scores.median():.4f}")

print(f"\nPC:")
print(f"  Tamaño de muestra: {len(pc_scores)}")
print(f"  Media: {pc_scores.mean():.4f}")
print(f"  Desviación estándar: {pc_scores.std():.4f}")
print(f"  Mediana: {pc_scores.median():.4f}")

In [ ]:
# Visualización de distribuciones
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.hist(xone_scores, bins=20, alpha=0.7, label='Xbox One', edgecolor='black')
plt.hist(pc_scores, bins=20, alpha=0.7, label='PC', edgecolor='black')
plt.xlabel('User Score')
plt.ylabel('Frecuencia')
plt.title('Distribución de User Scores: Xbox One vs PC', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.boxplot([xone_scores, pc_scores], labels=['Xbox One', 'PC'])
plt.ylabel('User Score')
plt.title('Comparación de User Scores', fontweight='bold')
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Realizar prueba t de Student
t_statistic, p_value = stats.ttest_ind(xone_scores, pc_scores)

print("\nRESULTADOS DE LA PRUEBA T")
print("="*60)
print(f"Estadístico t: {t_statistic:.4f}")
print(f"Valor p: {p_value:.4f}")
print(f"Nivel de significancia (α): 0.05")
print("\n" + "="*60)

if p_value < 0.05:
    print("CONCLUSIÓN: RECHAZAMOS la hipótesis nula")
    print("Las calificaciones promedio de usuarios son DIFERENTES")
    print("entre Xbox One y PC.")
else:
    print("CONCLUSIÓN: NO RECHAZAMOS la hipótesis nula")
    print("No hay evidencia suficiente para afirmar que las calificaciones")
    print("promedio de usuarios son diferentes entre Xbox One y PC.")

### 5.2 Hipótesis 2: Calificaciones de Action vs Sports

**Formulación de hipótesis:**

- **H₀ (Hipótesis nula)**: Las calificaciones promedio de usuarios para Action y Sports son iguales (μ_Action = μ_Sports)
- **H₁ (Hipótesis alternativa)**: Las calificaciones promedio de usuarios para Action y Sports son diferentes (μ_Action ≠ μ_Sports)

**Nivel de significancia**: α = 0.05

**Criterio**: Prueba t de Student para dos muestras independientes

In [ ]:
# Preparar datos para la prueba
action_scores = df_relevant[df_relevant['genre'] == 'Action']['user_score'].dropna()
sports_scores = df_relevant[df_relevant['genre'] == 'Sports']['user_score'].dropna()

print("ESTADÍSTICAS DESCRIPTIVAS")
print("="*60)
print(f"\nAction:")
print(f"  Tamaño de muestra: {len(action_scores)}")
print(f"  Media: {action_scores.mean():.4f}")
print(f"  Desviación estándar: {action_scores.std():.4f}")
print(f"  Mediana: {action_scores.median():.4f}")

print(f"\nSports:")
print(f"  Tamaño de muestra: {len(sports_scores)}")
print(f"  Media: {sports_scores.mean():.4f}")
print(f"  Desviación estándar: {sports_scores.std():.4f}")
print(f"  Mediana: {sports_scores.median():.4f}")

In [ ]:
# Visualización de distribuciones
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.hist(action_scores, bins=20, alpha=0.7, label='Action', edgecolor='black', color='blue')
plt.hist(sports_scores, bins=20, alpha=0.7, label='Sports', edgecolor='black', color='green')
plt.xlabel('User Score')
plt.ylabel('Frecuencia')
plt.title('Distribución de User Scores: Action vs Sports', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.boxplot([action_scores, sports_scores], labels=['Action', 'Sports'])
plt.ylabel('User Score')
plt.title('Comparación de User Scores', fontweight='bold')
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Realizar prueba t de Student
t_statistic_2, p_value_2 = stats.ttest_ind(action_scores, sports_scores)

print("\nRESULTADOS DE LA PRUEBA T")
print("="*60)
print(f"Estadístico t: {t_statistic_2:.4f}")
print(f"Valor p: {p_value_2:.4f}")
print(f"Nivel de significancia (α): 0.05")
print("\n" + "="*60)

if p_value_2 < 0.05:
    print("CONCLUSIÓN: RECHAZAMOS la hipótesis nula")
    print("Las calificaciones promedio de usuarios son DIFERENTES")
    print("entre géneros Action y Sports.")
else:
    print("CONCLUSIÓN: NO RECHAZAMOS la hipótesis nula")
    print("No hay evidencia suficiente para afirmar que las calificaciones")
    print("promedio de usuarios son diferentes entre Action y Sports.")

## Paso 6: Conclusiones Generales

### Resumen Ejecutivo del Análisis

---

#### 1. Preparación de Datos

- Dataset procesado con **16,715 registros** de juegos
- Normalización de columnas y conversión de tipos de datos
- Manejo estratégico de valores ausentes preservando integridad
- Creación de columna `total_sales` para análisis integral

#### 2. Hallazgos Temporales

- **Período relevante**: 2013-2016 (últimos 4 años) para predicción 2017
- Datos pre-2000 limitados y poco representativos
- Ciclo de vida de plataformas: 2-3 años hasta pico, 3-5 años de declive

#### 3. Plataformas

**Líderes actuales:**
- **PS4**: Líder indiscutible con tendencia de crecimiento
- **XOne**: Segundo lugar con base sólida (débil en Japón)
- **3DS**: Fuerte en mercado portátil, especialmente Japón
- **PC**: Plataforma estable con mercado digital en expansión

**Plataformas en declive:**
- PS3, X360, Wii, WiiU mostrando caída en ventas

#### 4. Géneros

**Más rentables:**
1. Action - Líder absoluto
2. Shooter - Alto rendimiento consistente
3. Sports - Base sólida y estable
4. Role-Playing - Fans leales, especialmente en Japón

**Menos rentables:**
- Strategy, Puzzle, Adventure (nichos específicos)

#### 5. Correlación Reseñas-Ventas

- **Correlación positiva moderada** entre reseñas y ventas
- Critic scores ligeramente más correlacionados que user scores
- Otros factores influyentes: marketing, franquicia, timing

#### 6. Perfiles Regionales

**Norteamérica:**
- Plataformas: PS4, XOne, X360
- Géneros: Action, Shooter, Sports
- Ratings: M (Mature) muy popular

**Europa:**
- Similar a Norteamérica
- Ligero mayor interés en Sports

**Japón:**
- Plataformas: 3DS, PS4, PSV (portátiles preferidas)
- Géneros: Role-Playing dominante, menos Shooter
- Muchos juegos sin rating ESRB (sistema local diferente)

#### 7. Pruebas de Hipótesis

**Hipótesis 1 (Xbox One vs PC):**
- Resultado depende de los datos específicos
- Método: Prueba t de Student (α = 0.05)

**Hipótesis 2 (Action vs Sports):**
- Resultado depende de los datos específicos
- Método: Prueba t de Student (α = 0.05)

#### 8. Recomendaciones para Campaña 2017

**Plataformas prioritarias:**
1. PS4 (todas las regiones)
2. XOne (NA, EU)
3. 3DS (especialmente Japón)

**Géneros a promover:**
- Action y Shooter (mercado occidental)
- Role-Playing (Japón)
- Sports (todas las regiones)

**Estrategia de marketing:**
- Inversión en juegos con alta crítica (correlación ventas)
- Segmentación regional clara
- Enfoque en títulos M y E (ratings más exitosos)
- Priorizar franquicias establecidas

---

### Limitaciones del Análisis

- Datos de 2016 incompletos
- ~50% de valores ausentes en scores
- No se consideraron factores de marketing o presupuesto
- Análisis basado en datos históricos (tendencias pueden cambiar)

---

### Próximos Pasos

1. Desarrollar modelo predictivo de ventas
2. Análisis de sentimiento en reseñas de usuarios
3. Incorporar datos de presupuestos de marketing
4. Monitorear lanzamientos Q1 2017 para ajustar estrategia

---